# Llama Track A: Frozen Retrieval Prompts

This notebook independently recreates all 60 accepted hard-retrieval prompts in human-readable English and downloads them as a Markdown document. It needs an internet connection to download the pinned tokenizer, but does **not** need model weights, CUDA, or a T4 GPU.

Run the cells in order. The final cell aborts unless the reconstructed prompt set matches the accepted SHA-256 fingerprint.

In [ ]:
# No GPU is required. This installs only the tokenizer library.
%pip -q install transformers


In [ ]:
import hashlib
import json
import random
from pathlib import Path

from transformers import AutoTokenizer

MODEL = 'casperhansen/llama-3.2-3b-instruct-awq'
REVISION = '272b3bde867b606760447deb9a4d2719fbdfd3ae'
ACCEPTED_DATE = '03 Aug 2026'
EXPECTED_SEMANTIC_SHA256 = 'fffa82607af6e7d2c892d18d5bc790524570833472ad397b7ab2e54eb03005ef'
EXPECTED_PROMPT_SET_SHA256 = '4e1ced4fd4829629929a346879cc00c45476f38fa0936137909b8779242321e8'
CONTEXT_TOKENS = 4096
DEPTHS = (0.1, 0.5, 0.9)
KEYS = ('AURORA', 'BOREAL', 'CEDAR', 'DRAGON', 'EMBER', 'FALCON', 'GARNET', 'HARBOR', 'INDIGO', 'JASPER', 'KESTREL', 'LANTERN', 'MARINER', 'NEBULA', 'ORBIT', 'PRAIRIE', 'QUARTZ', 'RAVEN', 'SUMMIT', 'TEMPEST', 'UMBRA', 'VECTOR', 'WILLOW', 'XENON', 'YARROW', 'ZEPHYR', 'ORBITAL', 'MARINA', 'CEDRIC', 'FALCONE', 'GARNET-X', 'HARBOR-X')
TOPICS = ('thermal scheduling', 'memory allocation', 'coastal monitoring', 'warehouse routing', 'battery maintenance', 'traffic forecasting', 'compiler testing', 'water treatment', 'satellite imaging', 'clinical logistics', 'network telemetry', 'acoustic sensing', 'grid balancing', 'crop analysis', 'robot calibration', 'archive indexing')
METHODS = ('compares recent measurements with a rolling baseline', 'records each decision before the next review cycle', 'uses independent checks to detect inconsistent observations', 'separates routine maintenance from exceptional interventions', 'prioritizes stable operation over short-lived peak performance', 'tracks uncertainty alongside every summarized measurement', 'repeats validation after any material configuration change', 'keeps raw observations available for later auditing')
OUTCOMES = ('The review found that seasonal variation explained most of the change.', 'Operators retained the conservative threshold for the next evaluation.', 'No single measurement was treated as sufficient evidence for deployment.', 'The team documented both successful trials and inconclusive observations.', 'A later audit confirmed the ordering but revised the uncertainty estimate.', 'The report emphasized reproducibility and clearly separated derived values.', 'Follow-up work focused on failure recovery rather than headline throughput.', 'The final recommendation required another independent measurement period.')

def canonical_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=True).encode('utf-8')).hexdigest()

def ids_hash(ids):
    return hashlib.sha256(','.join(map(str, ids)).encode('ascii')).hexdigest()

def encode(tokenizer, text):
    return list(tokenizer.encode(text, add_special_tokens=False))

def record_text(record):
    return f"\n[Access record] System {record['key']} uses activation code {record['code']}.\n"

def make_scenarios():
    scenarios = []
    for depth_index, depth in enumerate(DEPTHS):
        for index in range(20):
            seed = 407_100 + depth_index * 100_000 + index
            rng = random.Random(seed)
            keys = [f'NODE-{key}' for key in KEYS]
            rng.shuffle(keys)
            codes = set()
            while len(codes) < 32:
                codes.add(f"{rng.choice('ABCDEFGH')}{rng.randrange(100, 1000):03d}-{rng.choice('KLMNPQ')}{rng.randrange(10, 100):02d}")
            codes = sorted(codes)
            rng.shuffle(codes)
            target_index = rng.randrange(32)
            records = [{'key': key, 'code': code, 'is_target': position == target_index} for position, (key, code) in enumerate(zip(keys, codes))]
            rng.shuffle(records)
            target = next(record for record in records if record['is_target'])
            scenarios.append({'scenario_id': f'd{int(depth * 100):02d}-s{index:02d}', 'seed': seed, 'depth': depth, 'target_key': target['key'], 'target_code': target['code'], 'records': records})
    manifest = {'manifest_schema': 'cse407-hard-retrieval-manifest-v1', 'generator_version': 'track-a-hard-retrieval-v4-controlled-pairs-draft', 'status': 'DRAFT_UNTIL_C1W_C2_PILOT_PASSES', 'base_seed': 407100, 'depths': list(DEPTHS), 'scenarios_per_depth': 20, 'record_count': 32, 'scenarios': scenarios}
    manifest['semantic_sha256'] = canonical_hash(manifest)
    assert manifest['semantic_sha256'] == EXPECTED_SEMANTIC_SHA256
    return scenarios

def filler_text(rng):
    paragraphs, topics, methods, outcomes = [], list(TOPICS), list(METHODS), list(OUTCOMES)
    for index in range(512):
        if index % len(topics) == 0: rng.shuffle(topics)
        if index % len(methods) == 0: rng.shuffle(methods)
        if index % len(outcomes) == 0: rng.shuffle(outcomes)
        paragraphs.append(f'Memo {index + 1} discusses {topics[index % len(topics)]}. The working group {methods[index % len(methods)]}. Sample batch {2000 + index} was reviewed by {2 + index % 7} analysts. {outcomes[index % len(outcomes)]}\n')
    return '\n'.join(paragraphs)

def pack(tokenizer, records, budget, rng):
    encoded_records = [encode(tokenizer, record_text(record)) for record in records]
    filler_budget = budget - sum(map(len, encoded_records))
    gaps = [16] * (len(records) + 1)
    order = list(range(len(gaps)))
    rng.shuffle(order)
    for index in range(filler_budget - sum(gaps)): gaps[order[index % len(gaps)]] += 1
    text = filler_text(rng)
    ids = encode(tokenizer, text)
    while len(ids) < filler_budget:
        text += filler_text(rng)
        ids = encode(tokenizer, text)
    result, cursor = [], 0
    for index, record_ids in enumerate(encoded_records):
        next_cursor = cursor + gaps[index]
        result.extend(ids[cursor:next_cursor])
        result.extend(record_ids)
        cursor = next_cursor
    result.extend(ids[cursor:cursor + gaps[-1]])
    assert len(result) == budget
    return result

def make_prompt(tokenizer, scenario):
    question = f"What is the code for {scenario['target_key']}?"
    messages = [{'role': 'system', 'content': 'Use only the document. Return exactly one activation code and no explanation.'}, {'role': 'user', 'content': f'<DOC>\n{{HARD_RETRIEVAL_DOCUMENT}}\n</DOC>\n{question}'}]
    template = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, date_string=ACCEPTED_DATE)
    prefix, suffix = template.split('{HARD_RETRIEVAL_DOCUMENT}')
    prefix_ids, suffix_ids = encode(tokenizer, prefix), encode(tokenizer, suffix)
    budget = CONTEXT_TOKENS - len(prefix_ids) - len(suffix_ids)
    target = next(record for record in scenario['records'] if record['is_target'])
    target_ids = encode(tokenizer, record_text(target))
    target_start = round(scenario['depth'] * budget)
    distractors = [record for record in scenario['records'] if not record['is_target']]
    rng = random.Random(scenario['seed'] + 17)
    rng.shuffle(distractors)
    before_count = round(scenario['depth'] * len(distractors))
    ids = prefix_ids + pack(tokenizer, distractors[:before_count], target_start, random.Random(scenario['seed'] + 31)) + target_ids + pack(tokenizer, distractors[before_count:], budget - target_start - len(target_ids), random.Random(scenario['seed'] + 47)) + suffix_ids
    assert len(ids) == CONTEXT_TOKENS
    return {'scenario_id': scenario['scenario_id'], 'depth': scenario['depth'], 'target_key': scenario['target_key'], 'target_code': scenario['target_code'], 'input_ids_sha256': ids_hash(ids), 'input_ids': ids}


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION, trust_remote_code=True)
prompts = [make_prompt(tokenizer, scenario) for scenario in make_scenarios()]
prompt_set_hash = canonical_hash([{'scenario_id': prompt['scenario_id'], 'sha256': prompt['input_ids_sha256']} for prompt in prompts])
assert prompt_set_hash == EXPECTED_PROMPT_SET_SHA256, 'Prompt hash mismatch: do not use this output.'

lines = ['# Frozen Llama Track A Hard-Retrieval Prompts', '', f'- Tokenizer: `{MODEL}`', f'- Tokenizer revision: `{REVISION}`', f'- Semantic SHA-256: `{EXPECTED_SEMANTIC_SHA256}`', f'- Prompt-set SHA-256: `{prompt_set_hash}`', '- Every prompt has exactly 4,096 input tokens.']
for prompt in prompts:
    text = tokenizer.decode(prompt['input_ids'], skip_special_tokens=False, clean_up_tokenization_spaces=False)
    lines.extend(['', f"## {prompt['scenario_id']}", '', f"- Target depth: {prompt['depth']}", f"- Target key: `{prompt['target_key']}`", f"- Expected activation code: `{prompt['target_code']}`", f"- Input-ID SHA-256: `{prompt['input_ids_sha256']}`", '', '```text', text, '```'])

output_path = Path('llama_hard_retrieval_prompts.md')
output_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print(f'Validated and wrote {len(prompts)} English prompts to {output_path}')

try:
    from google.colab import files
except ImportError:
    print(f'Download {output_path} from the current directory.')
else:
    files.download(str(output_path))
